# 05. Agentic RAG -- LangGraph로 직접 구축

Retrieval-Augmented Generation(RAG)을 세 가지 방법으로 구현합니다: LangChain RAG Agent, LangChain RAG Chain, 그리고 LangGraph StateGraph 기반 커스텀 RAG. 문서 관련성 평가, 쿼리 리라이트, 조건부 라우팅 등 심화 패턴을 다룹니다.

## 학습 목표

- RAG 파이프라인(인덱싱 -> 검색 -> 생성)의 전체 구조를 이해한다
- `RecursiveCharacterTextSplitter`로 문서를 청킹한다
- `InMemoryVectorStore`로 벡터 스토어를 구축한다
- LangChain `create_agent` + `@tool`로 RAG Agent를 구현한다
- `@dynamic_prompt` 미들웨어로 RAG Chain(단일 LLM 호출)을 구현한다
- LangGraph `StateGraph`로 커스텀 RAG 에이전트를 구축한다
- `GradeDocuments` 구조화 출력으로 문서 관련성을 평가한다
- 쿼리 리라이트와 조건부 라우팅을 구현한다

## 5.1 환경 설정

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4.1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 5.2 RAG 개요

RAG(Retrieval-Augmented Generation)는 외부 지식을 검색하여 LLM 응답의 정확도를 높이는 패턴입니다. LLM은 두 가지 핵심 제약을 가집니다:
- **유한한 컨텍스트**: 전체 코퍼스를 한 번에 처리할 수 없음
- **정적 지식**: 학습 데이터가 시간이 지나면 구식이 됨

RAG는 쿼리 시점에 관련 외부 정보를 가져와 이 제약을 극복합니다.

### 파이프라인: 인덱싱 -> 검색 -> 생성

```
[문서] -> Text Splitter -> [청크] -> Embeddings -> [벡터 스토어]
                                                      |
[질문] -> Embedding -> similarity_search -> [관련 청크] -> LLM -> [답변]
```

### 5가지 핵심 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **Document Loaders** | 외부 소스(Google Drive, Notion 등)에서 데이터를 표준 Document 객체로 수집 |
| **Text Splitters** | 대규모 문서를 컨텍스트 윈도우에 맞는 청크로 분할 |
| **Embedding Models** | 텍스트를 의미적으로 유사한 내용이 가까이 모이는 벡터로 변환 |
| **Vector Stores** | 임베딩을 저장하고 유사도 검색을 수행하는 전문 데이터베이스 |
| **Retrievers** | 비정형 쿼리를 기반으로 관련 문서를 반환 |

### 세 가지 RAG 아키텍처

| 접근법 | 아키텍처 | LLM 호출 | 유연성 | 적합한 경우 |
|---|---|---|---|---|
| **2-Step RAG** | 검색 후 즉시 생성 | 단일 | 낮음 | FAQ, 문서 봇 (빠르고 예측 가능) |
| **Agentic RAG** | 에이전트가 검색 시점/방법 결정 | 다중 | 높음 | 복잡한 리서치, 다중 도구 접근 |
| **Hybrid RAG** | 쿼리 강화 + 검색 검증 + 답변 품질 체크 | 다중 | 높음 | 반복적 정제가 필요한 경우 |

### Agent vs Chain 접근 (LangChain 구현)

| 접근법 | 아키텍처 | LLM 호출 | 적합한 경우 |
|---|---|---|---|
| **RAG Agent** | 에이전트 + retriever 도구 | 다중 | 복잡한 쿼리, 쿼리 재구성 필요 |
| **RAG Chain** | 미들웨어 주입 컨텍스트 | 단일 | 단순 Q&A, 예측 가능한 비용 |
| **LangGraph 커스텀** | StateGraph + 커스텀 노드 | 다중 | 관련성 평가, 리라이트 등 세밀한 제어 |

## 5.3 문서 로딩 & 청킹

### 문서 로더 (Document Loaders)
문서 로더는 다양한 소스에서 원시 콘텐츠를 읽어 `page_content`와 `metadata` 필드를 가진 `Document` 객체로 반환합니다.

| 로더 | 소스 | 패키지 |
|---|---|---|
| `PyPDFLoader` | PDF 파일 | `pypdf` |
| `TextLoader` | 텍스트 파일 | 내장 |
| `CSVLoader` | CSV 파일 | 내장 |
| `WebBaseLoader` | 웹 페이지 | `beautifulsoup4` |
| `DirectoryLoader` | 디렉토리 내 파일들 | 내장 |

### 텍스트 분할 (Text Splitting)
`RecursiveCharacterTextSplitter`는 `\n\n` -> `\n` -> ` ` -> `""` 순으로 재귀적으로 분할하여 의미적 연관성을 유지합니다. 가장 범용적인 분할기로 권장됩니다.

| 파라미터 | 설명 | 권장값 |
|---|---|---|
| `chunk_size` | 청크 최대 문자 수 | 500-2000 (작으면 정밀 검색, 크면 맥락 보존) |
| `chunk_overlap` | 인접 청크 공유 문자 수 | chunk_size의 10-20% (경계 정보 손실 방지) |

### 기타 분할기

| 분할기 | 적합한 경우 |
|---|---|
| `MarkdownHeaderTextSplitter` | 마크다운 문서 |
| `HTMLHeaderTextSplitter` | HTML 문서 |
| `TokenTextSplitter` | 토큰 예산 기반 분할 |
| `CodeTextSplitter` | 소스 코드 (언어 인식) |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

raw_docs = [
    Document(
        page_content="LangGraph는 LLM으로 상태 기반 멀티 액터 "
        "애플리케이션을 구축하기 위한 프레임워크입니다.",
        metadata={"source": "langgraph-docs"},
    ),
    Document(
        page_content="에이전트는 도구를 사용하여 외부 시스템과 "
        "상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다.",
        metadata={"source": "agent-guide"},
    ),
]
print(f"문서 {len(raw_docs)}개 로드됨.")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = text_splitter.split_documents(raw_docs)

for i, doc in enumerate(splits):
    print(f"청크 {i}: {doc.page_content[:60]}...")
print(f"총 청크 수: {len(splits)}")

## 5.4 벡터 스토어 구축

벡터 스토어는 임베딩을 인덱싱하고 유사도 검색을 수행하는 전문 데이터베이스입니다. `InMemoryVectorStore`는 개발/테스트용으로 적합합니다.

### 주요 벡터 스토어 비교

| 벡터 스토어 | 유형 | 적합한 경우 |
|---|---|---|
| `InMemoryVectorStore` | 인프로세스 | 개발, 소규모 데이터셋 |
| `Chroma` | 임베디드/클라이언트-서버 | 프로토타이핑, 중규모 데이터셋 |
| `FAISS` | 인프로세스 | 고성능 로컬 검색 |
| `Pinecone` | 매니지드 클라우드 | 프로덕션, 확장성 |
| `PGVector` | PostgreSQL 확장 | 기존 PostgreSQL 인프라 활용 |

### 검색 유형

| 검색 타입 | 설명 |
|---|---|
| `"similarity"` | 표준 최근접 이웃 검색 |
| `"mmr"` | Maximal Marginal Relevance -- 관련성과 다양성의 균형 (중복 감소) |
| `"similarity_score_threshold"` | 최소 유사도 점수 이상인 문서만 반환 |

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
)
test_results = vector_store.similarity_search("LangGraph", k=2)
for doc in test_results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:80]}")
print(f"벡터 스토어 준비 완료. 문서 {len(splits)}개.")

## 5.5 검색 도구 정의

`response_format="content_and_artifact"`를 사용하면 도구 출력을 두 부분으로 분리합니다:
- **Content**: 모델에 전달되는 문자열 표현 (추론에 사용)
- **Artifact**: 원본 Document 객체 (프로그래밍 방식으로 접근 가능하지만 모델에 전송되지 않음)

이 분리를 통해 모델에는 읽기 쉬운 텍스트를, 후속 처리에는 메타데이터가 포함된 원본 객체를 사용할 수 있습니다.

In [ ]:
from langchain_core.tools import tool


@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """지식 베이스에서 관련 문서를 검색합니다."""
    docs = vector_store.similarity_search(query, k=4)
    serialized = "\n\n".join(
        f"출처: {d.metadata.get('source', '?')}\n{d.page_content}" for d in docs
    )
    return serialized, docs

## 5.6 LangChain RAG Agent -- `create_agent` + `@tool`

가장 간단한 방법: retriever를 도구로 등록하고 에이전트가 필요할 때 호출합니다.

### 다중 단계 검색 흐름
RAG Agent는 자동으로 다중 검색 단계를 실행할 수 있습니다:
1. **초기 검색** -- 사용자 질문 기반 쿼리 생성
2. **결과 평가** -- 검색된 문서가 질문에 충분한지 판단
3. **재구성 및 재검색** -- 결과가 부족하면 쿼리를 수정하여 재검색
4. **통합** -- 모든 검색 결과를 결합하여 최종 답변 생성

이 접근법은 복잡한 리서치 질문에 적합하지만, 여러 번의 LLM 호출로 비용과 지연이 증가합니다.

In [ ]:
from langchain.agents import create_agent

rag_agent = create_agent(
    model=llm,
    tools=[retrieve],
    system_prompt="당신은 리서치 어시스턴트입니다. "
    "답변하기 전에 retrieve를 사용하여 검색하세요.",
)
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph란 무엇인가요?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)

## 5.7 LangChain RAG Chain -- `@dynamic_prompt` 미들웨어

단일 LLM 호출로 RAG를 구현합니다. `@dynamic_prompt`가 LLM 호출 전에 문서를 검색하고 시스템 프롬프트에 자동으로 주입합니다. 미들웨어 방식이므로 에이전트 루프 없이 **단일 패스**로 동작합니다.

| 특성 | RAG Agent | RAG Chain |
|---|---|---|
| LLM 호출 수 | 다중 (에이전트 결정) | 단일 |
| 검색 횟수 | 1회 이상 (에이전트 제어) | 정확히 1회 (미들웨어 제어) |
| 쿼리 재구성 | 자동 | 미지원 |
| 지연 | 높음 (여러 왕복) | 낮음 (단일 패스) |
| 비용 | 높음 (더 많은 토큰) | 낮음 (더 적은 토큰) |
| 투명성 | 에이전트 추론이 메시지에 노출 | 컨텍스트 주입이 암묵적 |

**고급 활용**: `@dynamic_prompt`로 기본 컨텍스트를 주입하면서 동시에 retriever 도구를 제공하여 두 접근법을 결합할 수도 있습니다.

In [ ]:
from langchain.agents.middleware import dynamic_prompt


@dynamic_prompt
def rag_prompt(request):
    """문서를 검색하여 시스템 프롬프트에 주입합니다."""
    user_msg = request.state["messages"][-1].content
    docs = vector_store.similarity_search(user_msg, k=4)
    ctx = "\n\n".join(d.page_content for d in docs)
    return f"컨텍스트를 기반으로 답변하세요:\n\n{ctx}"

In [ ]:
chain = create_agent(model=llm, middleware=[rag_prompt])
resp = chain.invoke(
    {"messages": [{"role": "user", "content": "에이전트는 어떻게 작동하나요?"}]},
    config=lf_config,
)
print(resp["messages"][-1].content)

## 5.8 LangGraph 커스텀 RAG -- StateGraph 구축

LangGraph `StateGraph`로 세밀한 제어가 가능한 RAG 에이전트를 직접 구축합니다. 이 방식의 핵심 장점은 **조건부 라우팅**을 통해 검색 결과의 관련성을 평가하고, 관련 없는 경우 쿼리를 리라이트하여 재검색하는 등의 세밀한 흐름 제어가 가능하다는 것입니다.

### 아키텍처

```
        [generate_query_or_respond]
             /              \
       (tool call)       (no tool call)
           |                  |
      [retrieve]           [END]
           |
   [grade_documents]
      /          \
(relevant)    (not relevant)
    |              |
[generate]   [rewrite_question]
    |              |
  [END]    [generate_query_or_respond]
```

### 각 노드의 역할

| 노드 | 역할 |
|---|---|
| `generate_query_or_respond` | 진입 노드. 검색할지 직접 응답할지 결정 |
| `retrieve` | `ToolNode`로 검색 실행 |
| `grade_documents` | 구조화 출력(`GradeDocuments`)으로 문서 관련성 평가 |
| `rewrite_question` | 관련 없는 결과 시 더 구체적인 쿼리로 리라이트 |
| `generate_answer` | 관련 문서 기반 최종 답변 생성 |

### 무한 루프 방지
`rewrite_question` -> `generate_query_or_respond` 순환이 발생할 수 있으므로, `retry_count`를 State에 추가하여 최대 재시도 횟수를 제한하는 것이 권장됩니다.

In [ ]:
from langgraph.graph import MessagesState


class AgentState(MessagesState):
    """커스텀 RAG 에이전트 상태."""

    relevance: str  # "relevant" or "not_relevant"


print(f"AgentState 키: {list(AgentState.__annotations__)}")

## 5.9 `generate_query_or_respond` 노드

진입 노드입니다. LLM이 retrieve 도구를 호출할지, 직접 응답할지 결정합니다.

In [ ]:
llm_with_tools = llm.bind_tools([retrieve])


def generate_query_or_respond(state: AgentState):
    """검색할지 직접 응답할지 결정합니다."""
    system = (
        "당신은 유용한 어시스턴트입니다. "
        "지식 베이스 관련 질문에는 retrieve 도구를 사용하세요."
    )
    msgs = [{"role": "system", "content": system}] + state["messages"]
    response = llm_with_tools.invoke(msgs, config=lf_config)
    return {"messages": [response]}

## 5.10 `grade_documents` 노드 -- 구조화 출력으로 관련성 평가

`GradeDocuments` 스키마로 LLM이 문서 관련성을 평가합니다. `with_structured_output`으로 구조화된 응답을 받습니다.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class GradeDocuments(BaseModel):
    """검색된 문서의 이진 관련성 점수."""

    relevance: Literal["relevant", "not_relevant"] = Field(
        description="문서가 관련이 있는지 여부."
    )
    reasoning: str = Field(description="간략한 설명.")


grader = llm.with_structured_output(GradeDocuments)

In [ ]:
def grade_documents(state: AgentState):
    """
    검색된 문서의 관련성을 평가합니다.
    """

    msgs = state["messages"]

    user_q = next((m.content for m in msgs if m.type == "human"), "")

    tool_content = msgs[-1].content

    grade = grader.invoke(
        f"질문: {user_q}\n문서:\n{tool_content}\n이 문서들이 관련이 있습니까?"
    )

    return {"relevance": grade.relevance, "messages": msgs}

## 5.11 `rewrite_question` 노드

검색된 문서가 관련 없을 때, 원래 질문을 더 구체적으로 리라이트하여 검색 품질을 향상시킵니다.

In [ ]:
def rewrite_question(state: AgentState):
    """
    검색 품질 향상을 위해 질문을 리라이트합니다.
    """

    user_q = next((m.content for m in state["messages"] if m.type == "human"), "")

    prompt = f"다른 용어를 사용하여 리라이트하세요.\n원본: {user_q}\n리라이트:"

    response = llm.invoke(prompt, config=lf_config)

    return {"messages": [{"role": "human", "content": response.content}]}

## 5.12 `generate_answer` 노드

관련 문서가 확인되면, 검색 결과와 원본 질문을 결합하여 최종 답변을 생성합니다.

In [ ]:
def generate_answer(state: AgentState):
    """
    검색된 문서를 사용하여 답변을 생성합니다.
    """

    user_q = next((m.content for m in state["messages"] if m.type == "human"), "")

    docs = next((m.content for m in state["messages"] if m.type == "tool"), "")

    prompt = f"컨텍스트만을 사용하여 답변하세요.\n컨텍스트:\n{docs}\n\n질문: {user_q}"

    resp = llm.invoke(prompt, config=lf_config)

    return {"messages": [{"role": "assistant", "content": resp.content}]}

## 5.13 그래프 조립 & 실행

모든 노드를 `StateGraph`에 등록하고, 조건부 엣지(`tools_condition`, `relevance_router`)로 연결합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def relevance_router(state: AgentState):
    if state.get("relevance") == "relevant":
        return "generate_answer"
    return "rewrite_question"


graph = StateGraph(AgentState)
graph.add_node("gen_query", generate_query_or_respond)

In [ ]:
graph.add_node("retrieve", ToolNode([retrieve]))
graph.add_node("grade_documents", grade_documents)
graph.add_node("rewrite_question", rewrite_question)
graph.add_node("generate_answer", generate_answer)

graph.add_edge(START, "gen_query")
graph.add_conditional_edges(
    "gen_query",
    tools_condition,
    {"tools": "retrieve", "__end__": END},
)

In [ ]:
graph.add_edge("retrieve", "grade_documents")
graph.add_conditional_edges(
    "grade_documents",
    relevance_router,
    {"generate_answer": "generate_answer", "rewrite_question": "rewrite_question"},
)
graph.add_edge("rewrite_question", "gen_query")
graph.add_edge("generate_answer", END)

app = graph.compile()
print("그래프 컴파일 성공.")

In [ ]:
for event in app.stream(
    {"messages": [{"role": "user", "content": "LangGraph란 무엇인가요?"}]},
    config=lf_config,
):
    for node_name, output in event.items():
        print(f"--- {node_name} ---")
        if "messages" in output:
            last = output["messages"][-1]
            txt = last.content if hasattr(last, "content") else str(last)
            print(txt[:200])

## 요약

### 세 가지 RAG 접근법 비교

| 특성 | RAG Agent | RAG Chain | LangGraph 커스텀 |
|---|---|---|---|
| LLM 호출 수 | 다중 | 단일 | 다중 |
| 검색 횟수 | 에이전트 결정 | 정확히 1회 | 커스텀 |
| 쿼리 재구성 | 자동 | 미지원 | 명시적 노드 |
| 관련성 평가 | 암묵적 | 없음 | `GradeDocuments` |
| 제어 수준 | 낮음 | 낮음 | 높음 |
| 구현 복잡도 | 낮음 | 최저 | 높음 |

### 핵심 LangGraph 패턴

| 패턴 | 구현 |
|---|---|
| 조건부 라우팅 | `add_conditional_edges` + `tools_condition` |
| 구조화 출력 | `llm.with_structured_output(GradeDocuments)` |
| 도구 노드 | `ToolNode([retrieve])` |
| 루프 제어 | `rewrite_question` -> `gen_query` 순환 |

### 다음 단계
→ **[06_sql_agent.ipynb](./06_sql_agent.ipynb)**: SQL 에이전트를 만듭니다.
